In [1]:
import os
import re
import subprocess
import time
import pandas as pd
import plotly.graph_objects as go
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
from pyswmm import Simulation, Nodes
import swmmio
import traceback

# 🔧 Modify INP File
def modify_inp_file(inp_file_path, pump_settings_file, output_inp_file_path):
    df = pd.read_csv(pump_settings_file, index_col="Name")
    pumps_off = df[df["Status"] == 0].index.tolist()

    with open(inp_file_path, "r") as file:
        lines = file.readlines()

    in_pumps_section = False
    modified_lines = []

    for line in lines:
        if line.strip().upper() == "[PUMPS]":
            in_pumps_section = True
            modified_lines.append(line)
            continue
        elif line.strip().startswith("[") and in_pumps_section:
            in_pumps_section = False

        if in_pumps_section and not line.strip().startswith(";") and line.strip():
            parts = re.split(r"\s+", line.strip())
            if len(parts) == 8 and parts[0] == "NESPS_Pump4_2_OMID" and parts[3] == "NESPS_Pump4" and parts[4] == "_2_OMID":
                parts[3] = "NESPS_Pump4\x1f_2_OMID"
                del parts[4]
            if len(parts) >= 7 and parts[0] in pumps_off:
                #print(f"🔧 Modifying pump: {parts[0]} → Startup depth = 1000")
                parts[5] = "1000"
                line = " ".join(parts) + "\n"
        modified_lines.append(line)

    report_lines = [
        "[REPORT]\n",
        "INPUT      YES\n",
        "CONTROLS   YES\n",
        "NODES      ALL\n",
        "LINKS      ALL\n"
    ]

    in_report_section = False
    report_found = False
    new_lines = []

    for line in modified_lines:
        if line.strip().upper() == "[REPORT]":
            in_report_section = True
            report_found = True
            new_lines.extend(report_lines)
            continue
        if in_report_section and line.strip().startswith("["):
            in_report_section = False
        if not in_report_section:
            new_lines.append(line)

    if not report_found:
        new_lines.append("\n")
        new_lines.extend(report_lines)

    with open(output_inp_file_path, "w") as file:
        file.writelines(new_lines)
    print(f"✅ Modified INP saved: {output_inp_file_path.name}")

# ▶ Run SWMM
def run_swmm_subprocess(swmm_exe, inp_file, rpt_file, out_file):
    try:
        proc = subprocess.run(
            [swmm_exe, str(inp_file), str(rpt_file), str(out_file)],
            stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True
        )
        if proc.returncode != 0:
            raise RuntimeError(proc.stderr)
        print(f"✔ SWMM completed: {inp_file.name}")
        return inp_file.stem
    except Exception as e:
        return f"❌ Failed: {inp_file.name} → {e}"

# 📊 Extract Max Depths
def extract_max_depths_from_rpt(inp_file, rpt_file, output_csv_file):
    model = swmmio.Model(str(inp_file))
    model.rpt_path = str(rpt_file)
    if model.rpt is None or not hasattr(model.rpt, 'node_depth_summary') or model.rpt.node_depth_summary is None:
        print(f"⚠️ No depth data in {rpt_file.name}")
        return False
    df = model.rpt.node_depth_summary
    df = df[["MaxNodeDepth"]].copy()
    df.index.name = "Node"
    df.reset_index(inplace=True)
    df.rename(columns={"MaxNodeDepth": "Max_depth(SWMM)"}, inplace=True)
    df.to_csv(output_csv_file, index=False)
    print(f"📤 Saved depths: {output_csv_file.name}")
    return True

# 🧠 PySWMM Postprocessing
def extract_node_full_depths(inp_file):
    depths = []
    with Simulation(str(inp_file)) as sim:
        sim.step_advance(7200)
        for node in Nodes(sim):
            depths.append({"Node": node.nodeid, "Full_Depth": node.full_depth})
    return pd.DataFrame(depths)

def export_basement_flooding(inp_file, max_depth_csv, output_csv):
    full_depth_df = extract_node_full_depths(inp_file)
    max_depth_df = pd.read_csv(max_depth_csv)
    max_depth_df.rename(columns={max_depth_df.columns[0]: "Node"}, inplace=True)
    df = pd.merge(full_depth_df, max_depth_df, on="Node", how="left").fillna(0)
    df["basement flooding"] = df.apply(lambda r: 1 if r["Max_depth(SWMM)"] >= r["Full_Depth"] - 8 else 0, axis=1)
    df["Flooded_Depth"] = df.apply(lambda r: r["Max_depth(SWMM)"] - (r["Full_Depth"] - 8) if r["basement flooding"] else 0, axis=1)
    df.to_csv(output_csv, index=False)
    print(f"💾 Flooding results: {output_csv.name}")

# 🗺️ Plot Flooding Map
def read_coordinates(inp_file):
    coords = {}
    with open(inp_file, 'r') as file:
        lines = file.readlines()
    in_coords = False
    for line in lines:
        if line.strip().upper() == '[COORDINATES]':
            in_coords = True
            continue
        if in_coords and line.strip().startswith('['):
            break
        if in_coords and line.strip() and not line.strip().startswith(';'):
            parts = line.split()
            if len(parts) >= 3:
                coords[parts[0]] = (float(parts[1]), float(parts[2]))
    return coords

def plot_flooding_map(inp_file, flood_csv, tag):
    coords = read_coordinates(inp_file)
    df = pd.read_csv(flood_csv)
    df['Node'] = df['Node'].astype(str)
    flooded = df[(df['basement flooding'] == 1) & (df['Node'].isin(coords))]
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=[coords[n][0] for n in coords],
        y=[coords[n][1] for n in coords],
        mode='markers', marker=dict(color='lightgrey', size=4), name='All Nodes', hoverinfo='skip'))
    fig.add_trace(go.Scatter(
        x=[coords[n][0] for n in flooded['Node']],
        y=[coords[n][1] for n in flooded['Node']],
        mode='markers', marker=dict(color='red', size=7),
        name='Flooded Nodes', text=flooded['Node'], hoverinfo='text'))
    fig.update_layout(title=f"Flooding Map: {tag}", height=700, width=900)
    fig.write_html(str(inp_file.parent / f"{tag}_flooding_map.html"))
    print(f"🗺️ Map saved: {tag}_flooding_map.html")

# 🔁 MAIN PIPELINE (Force All)
def run_pipeline():
    base_path = Path(r"C:\Users\csofios\Desktop\LongTermCSO\SWMM")
    inp_file_path = base_path / "GLWA_Core_v4_R05_2020_p2_harris_no_event_2.inp"
    pump_folder = base_path / "pump_status_20"
    swmm_exe = r"C:\Program Files\EPA SWMM 5.2.4 (64-bit)\runswmm.exe"

    pump_files = sorted(pump_folder.glob("*.csv"))
    tasks = []

    for pump_file in pump_files:
        tag = pump_file.stem
        mod_inp = base_path / f"{tag}_modified.inp"
        rpt_file = base_path / f"{tag}_modified.rpt"
        out_file = base_path / f"{tag}_modified.out"
        modify_inp_file(inp_file_path, pump_file, mod_inp)
        tasks.append((swmm_exe, mod_inp, rpt_file, out_file, tag))

    completed_tags = []
    with ThreadPoolExecutor(max_workers=4) as executor:
        futures = [executor.submit(run_swmm_subprocess, *args[:4]) for args in tasks]
        for future, (_, _, _, _, tag) in zip(as_completed(futures), tasks):
            result = future.result()
            if result and not result.startswith("❌"):
                completed_tags.append(tag)
            print(result)

    for tag in completed_tags:
        try:
            mod_inp = base_path / f"{tag}_modified.inp"
            rpt_file = base_path / f"{tag}_modified.rpt"
            max_csv = base_path / f"{tag}_depths.csv"
            flood_csv = base_path / f"{tag}_flooding.csv"
            extract_max_depths_from_rpt(mod_inp, rpt_file, max_csv)
            export_basement_flooding(mod_inp, max_csv, flood_csv)
            plot_flooding_map(mod_inp, flood_csv, tag)
        except Exception as e:
            print(f"❌ Post-processing failed for {tag}: {e}")

if __name__ == "__main__":
    start = time.time()
    run_pipeline()
    print(f"\n✅ Total runtime: {(time.time() - start) / 60:.2f} minutes")


✅ Modified INP saved: pump_status_1_modified.inp
✅ Modified INP saved: pump_status_10_modified.inp
✅ Modified INP saved: pump_status_11_modified.inp
✅ Modified INP saved: pump_status_12_modified.inp
✅ Modified INP saved: pump_status_13_modified.inp
✅ Modified INP saved: pump_status_14_modified.inp
✅ Modified INP saved: pump_status_15_modified.inp
✅ Modified INP saved: pump_status_16_modified.inp
✅ Modified INP saved: pump_status_17_modified.inp
✅ Modified INP saved: pump_status_18_modified.inp
✅ Modified INP saved: pump_status_19_modified.inp
✅ Modified INP saved: pump_status_2_modified.inp
✅ Modified INP saved: pump_status_20_modified.inp
✅ Modified INP saved: pump_status_3_modified.inp
✅ Modified INP saved: pump_status_4_modified.inp
✅ Modified INP saved: pump_status_5_modified.inp
✅ Modified INP saved: pump_status_6_modified.inp
✅ Modified INP saved: pump_status_7_modified.inp
✅ Modified INP saved: pump_status_8_modified.inp
✅ Modified INP saved: pump_status_9_modified.inp
✔ SWMM co

In [2]:
import pandas as pd
import plotly.graph_objects as go
from pathlib import Path

def read_coordinates(inp_file):
    coords = {}
    with open(inp_file, 'r') as file:
        lines = file.readlines()
    in_coords = False
    for line in lines:
        if line.strip().upper() == '[COORDINATES]':
            in_coords = True
            continue
        if in_coords and line.strip().startswith('['):
            break
        if in_coords and line.strip() and not line.strip().startswith(';'):
            parts = line.split()
            if len(parts) >= 3:
                coords[parts[0]] = (float(parts[1]), float(parts[2]))
    return coords

def plot_unique_flooding(base_path, inp_file, tags):
    coords = read_coordinates(inp_file)

    flooded_sets = {}
    for tag in tags:
        df = pd.read_csv(base_path / f"{tag}_flooding.csv")
        flooded_nodes = set(df[df["basement flooding"] == 1]["Node"].astype(str))
        flooded_sets[tag] = flooded_nodes

    all_nodes = set(coords.keys())

    for tag in tags:
        unique_flooded = flooded_sets[tag].copy()
        for other_tag in tags:
            if other_tag != tag:
                unique_flooded -= flooded_sets[other_tag]

        fig = go.Figure()
        fig.add_trace(go.Scatter(
            x=[coords[n][0] for n in all_nodes if n in coords],
            y=[coords[n][1] for n in all_nodes if n in coords],
            mode='markers',
            marker=dict(color='lightgrey', size=4),
            name='All Nodes',
            hoverinfo='skip'))

        fig.add_trace(go.Scatter(
            x=[coords[n][0] for n in unique_flooded if n in coords],
            y=[coords[n][1] for n in unique_flooded if n in coords],
            mode='markers',
            marker=dict(color='green', size=7),
            name=f'Only Flooded in {tag}',
            text=list(unique_flooded),
            hoverinfo='text'))

        fig.update_layout(
            title=f"Unique Basement Flooding: {tag}",
            height=700,
            width=900
        )
        fig.write_html(str(base_path / f"{tag}_unique_flooding_map.html"))
        print(f"✅ Map saved: {tag}_unique_flooding_map.html")

if __name__ == "__main__":
    base_path = Path(r"C:\Users\csofios\Desktop\LongTermCSO\SWMM")
    inp_file = base_path / "GLWA_Core_v4_R05_2020_p2_harris_no_event_2.inp"
    tags = [f"pump_status_{i}" for i in range(1, 41)]
    plot_unique_flooding(base_path, inp_file, tags)


✅ Map saved: pump_status_1_unique_flooding_map.html
✅ Map saved: pump_status_2_unique_flooding_map.html
✅ Map saved: pump_status_3_unique_flooding_map.html
✅ Map saved: pump_status_4_unique_flooding_map.html
✅ Map saved: pump_status_5_unique_flooding_map.html
✅ Map saved: pump_status_6_unique_flooding_map.html
✅ Map saved: pump_status_7_unique_flooding_map.html
✅ Map saved: pump_status_8_unique_flooding_map.html
✅ Map saved: pump_status_9_unique_flooding_map.html
✅ Map saved: pump_status_10_unique_flooding_map.html
✅ Map saved: pump_status_11_unique_flooding_map.html
✅ Map saved: pump_status_12_unique_flooding_map.html
✅ Map saved: pump_status_13_unique_flooding_map.html
✅ Map saved: pump_status_14_unique_flooding_map.html
✅ Map saved: pump_status_15_unique_flooding_map.html
✅ Map saved: pump_status_16_unique_flooding_map.html
✅ Map saved: pump_status_17_unique_flooding_map.html
✅ Map saved: pump_status_18_unique_flooding_map.html
✅ Map saved: pump_status_19_unique_flooding_map.html
✅ 

In [3]:
import pandas as pd
import plotly.graph_objects as go
from pathlib import Path

# ─────────────────────────────────────────────────────────────
# Read Coordinates from INP File
# ─────────────────────────────────────────────────────────────
def read_coordinates(inp_file):
    coords = {}
    with open(inp_file, 'r') as file:
        lines = file.readlines()

    in_coords = False
    for line in lines:
        line = line.strip()
        if line.upper().startswith('[COORDINATES]'):
            in_coords = True
            continue
        if in_coords and line.startswith('['):
            break
        if in_coords and line and not line.startswith(';'):
            parts = line.split()
            if len(parts) >= 3:
                coords[parts[0]] = (float(parts[1]), float(parts[2]))
    return coords

# ─────────────────────────────────────────────────────────────
# Plot Flooding Map with Flooded Depth Capped at 8 ft
# ─────────────────────────────────────────────────────────────
def plot_flooding_map_colored(inp_file, flooding_csv, html_export_path, scenario_column="basement flooding"):
    coords = read_coordinates(inp_file)
    df = pd.read_csv(flooding_csv)
    df['Node'] = df['Node'].astype(str)

    # 🔒 Cap Flooded Depth at 8 ft
    df["Flooded_Depth"] = df["Flooded_Depth"].clip(upper=8)

    x_all, y_all = zip(*[coords[n] for n in coords])

    flooded = df[(df[scenario_column] == 1) & (df['Node'].isin(coords))].copy()
    if flooded.empty:
        print(f"⚠️ No flooded nodes in {flooding_csv}")
        return

    x_flooded = [coords[n][0] for n in flooded['Node']]
    y_flooded = [coords[n][1] for n in flooded['Node']]
    depths = flooded["Flooded_Depth"]
    labels = flooded.apply(lambda row: f"{row['Node']}<br>Flooded Depth: {round(row['Flooded_Depth'], 2)} ft", axis=1)

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=x_all, y=y_all,
        mode='markers',
        marker=dict(color='lightgrey', size=4),
        name='All Nodes',
        hoverinfo='skip'
    ))

    fig.add_trace(go.Scatter(
        x=x_flooded, y=y_flooded,
        mode='markers',
        marker=dict(
            color=depths,
            colorscale='Viridis',
            size=8,
            colorbar=dict(title='Flooded Depth (ft)'),
            cmin=0,
            cmax=8,
            showscale=True
        ),
        text=labels,
        hoverinfo='text',
        name='Flooded Nodes'
    ))

    fig.update_layout(
        title=html_export_path.stem.replace('_', ' ').title(),
        xaxis_title="X",
        yaxis_title="Y",
        height=700,
        width=1000,
        hovermode="closest"
    )

    fig.write_html(str(html_export_path))
    print(f"✅ Map saved to: {html_export_path}")

# ─────────────────────────────────────────────────────────────
# Generate Maps for All *_flooding.csv Files
# ─────────────────────────────────────────────────────────────
def generate_flood_maps():
    base_path = Path(r"C:\Users\csofios\Desktop\LongTermCSO\SWMM")
    flooding_files = sorted(base_path.glob("*_flooding.csv"))

    for flood_csv in flooding_files:
        tag = flood_csv.stem.replace("_flooding", "")
        inp_file = base_path / f"{tag}_modified.inp"
        html_file = base_path / f"{tag}_flooding_map.html"

        if not inp_file.exists():
            print(f"❌ Missing INP file: {inp_file}")
            continue

        plot_flooding_map_colored(inp_file, flood_csv, html_file)

# ─────────────────────────────────────────────────────────────
# Run Main
# ─────────────────────────────────────────────────────────────
if __name__ == "__main__":
    generate_flood_maps()


✅ Map saved to: C:\Users\csofios\Desktop\LongTermCSO\SWMM\pump_status_1_flooding_map.html
✅ Map saved to: C:\Users\csofios\Desktop\LongTermCSO\SWMM\pump_status_2_flooding_map.html
✅ Map saved to: C:\Users\csofios\Desktop\LongTermCSO\SWMM\pump_status_3_flooding_map.html
✅ Map saved to: C:\Users\csofios\Desktop\LongTermCSO\SWMM\pump_status_4_flooding_map.html
✅ Map saved to: C:\Users\csofios\Desktop\LongTermCSO\SWMM\pump_status_5_flooding_map.html
